# AgroBot — Crop Recommendation Model

**Project:** Smart Agriculture System for Nepal (FYP)
**Author:** Prashant Rijal

## Methodology

Since real field sensor data is not yet available (rover under construction), this notebook generates a synthetic training dataset based on published agronomic requirements for crops commonly grown in Nepal.

**Sources:**
- FAO Crop Water Requirements Guidelines
- Nepal Department of Agriculture — Crop Production Guidelines
- ICIMOD Mountain Agriculture Research

**Crops included:** Maize, Tomato, Rice, Wheat, Potato, Mustard

**Input features (from ESP32 rover sensors):**
- Nitrogen (mg/kg)
- Phosphorus (mg/kg)
- Potassium (mg/kg)
- Temperature (°C)
- Moisture (%)
- pH

**Output:** Recommended crop

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
print('Libraries loaded successfully.')

## 1. Define Agronomic Parameters

Each crop has an optimal range for each sensor reading.
We define `(min, max, mean, std)` for each feature per crop,
derived from agronomic guidelines for Nepal's conditions.

In [ ]:
# Format: (min, max, mean, std)
# All values reflect conditions in Nepal's agricultural zones

CROP_PARAMS = {
    'Maize': {
        'nitrogen':    (60,  140, 95,  18),
        'phosphorus':  (30,  60,  44,  8),
        'potassium':   (40,  85,  60,  12),
        'temperature': (18,  35,  25,  4),
        'moisture':    (50,  78,  63,  8),
        'ph':          (5.8, 7.0, 6.3, 0.3),
    },
    'Tomato': {
        'nitrogen':    (50,  110, 75,  15),
        'phosphorus':  (45,  85,  62,  10),
        'potassium':   (65,  125, 90,  15),
        'temperature': (16,  28,  22,  3),
        'moisture':    (58,  80,  68,  7),
        'ph':          (6.0, 7.0, 6.4, 0.25),
    },
    'Rice': {
        'nitrogen':    (40,  105, 70,  17),
        'phosphorus':  (20,  50,  34,  8),
        'potassium':   (25,  65,  44,  10),
        'temperature': (22,  38,  29,  4),
        'moisture':    (72,  96,  82,  7),
        'ph':          (5.5, 7.0, 6.1, 0.35),
    },
    'Wheat': {
        'nitrogen':    (45,  100, 68,  14),
        'phosphorus':  (35,  70,  50,  9),
        'potassium':   (35,  75,  52,  10),
        'temperature': (5,   22,  14,  4),
        'moisture':    (38,  62,  50,  7),
        'ph':          (6.0, 7.5, 6.7, 0.35),
    },
    'Potato': {
        'nitrogen':    (65,  130, 90,  17),
        'phosphorus':  (55,  100, 72,  12),
        'potassium':   (85,  135, 108, 14),
        'temperature': (10,  22,  16,  3),
        'moisture':    (60,  80,  70,  7),
        'ph':          (5.5, 6.5, 6.0, 0.25),
    },
    'Mustard': {
        'nitrogen':    (35,  85,  55,  13),
        'phosphorus':  (25,  55,  38,  8),
        'potassium':   (28,  60,  42,  9),
        'temperature': (8,   22,  15,  4),
        'moisture':    (38,  62,  48,  7),
        'ph':          (6.0, 7.5, 6.6, 0.35),
    },
}

FEATURES = ['nitrogen', 'phosphorus', 'potassium', 'temperature', 'moisture', 'ph']
CROPS = list(CROP_PARAMS.keys())
SAMPLES_PER_CROP = 500

print(f'Crops: {CROPS}')
print(f'Features: {FEATURES}')
print(f'Samples per crop: {SAMPLES_PER_CROP} → Total: {len(CROPS) * SAMPLES_PER_CROP}')

## 2. Generate Synthetic Dataset

In [ ]:
def generate_crop_samples(crop_name, params, n=500):
    rows = []
    for _ in range(n):
        row = {}
        for feat in FEATURES:
            mn, mx, mu, sigma = params[feat]
            val = np.random.normal(mu, sigma)
            val = np.clip(val, mn, mx)   # stay within agronomic bounds
            row[feat] = round(val, 2)
        row['crop'] = crop_name
        rows.append(row)
    return rows

all_rows = []
for crop, params in CROP_PARAMS.items():
    all_rows.extend(generate_crop_samples(crop, params, SAMPLES_PER_CROP))

df = pd.DataFrame(all_rows).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['crop'].value_counts())
df.head(10)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
colors = ['#2d6a4f','#e63946','#457b9d','#f4a261','#6a4c93','#2a9d8f']

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    for j, crop in enumerate(CROPS):
        data = df[df['crop'] == crop][feat]
        ax.hist(data, bins=25, alpha=0.6, label=crop, color=colors[j])
    ax.set_title(feat.capitalize(), fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Feature Distribution by Crop — Nepal Agricultural Conditions',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../figures and Diagrams/feature_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 5))
corr = df[FEATURES].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlGn',
            linewidths=0.5, square=True, cbar_kws={'shrink': .8})
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures and Diagrams/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Training — Random Forest Classifier

In [ ]:
X = df[FEATURES].values
y = df['crop'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')
print(f'Classes          : {le.classes_}')

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc  = accuracy_score(y_test,  model.predict(X_test))

print(f'Train Accuracy : {train_acc*100:.2f}%')
print(f'Test  Accuracy : {test_acc*100:.2f}%')

## 5. Model Evaluation

In [ ]:
# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y_enc, cv=cv, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print(f'Individual folds:   {[f"{s*100:.1f}%" for s in cv_scores]}')

In [ ]:
# Classification report
y_pred = model.predict(X_test)
print('Classification Report:')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5, cbar_kws={'shrink': .8})
plt.xlabel('Predicted', fontweight='bold')
plt.ylabel('Actual', fontweight='bold')
plt.title('Confusion Matrix — Crop Recommendation Model', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures and Diagrams/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
colors_bar = ['#2d6a4f' if v >= feat_imp.median() else '#74c69d' for v in feat_imp]
feat_imp.plot(kind='barh', color=colors_bar)
plt.xlabel('Importance Score', fontweight='bold')
plt.title('Feature Importance — Random Forest', fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures and Diagrams/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFeature Importances:')
for feat, imp in feat_imp.sort_values(ascending=False).items():
    print(f'  {feat:<14} {imp:.4f} ({imp*100:.1f}%)')

## 6. Test with Sample Inputs

In [ ]:
def predict_crop(nitrogen, phosphorus, potassium, temperature, moisture, ph):
    inp = np.array([[nitrogen, phosphorus, potassium, temperature, moisture, ph]])
    pred_enc = model.predict(inp)[0]
    proba    = model.predict_proba(inp)[0]
    crop     = le.inverse_transform([pred_enc])[0]
    conf     = round(float(proba[pred_enc]) * 100, 1)
    
    top3 = sorted(zip(le.classes_, proba), key=lambda x: -x[1])[:3]
    print(f'Recommended Crop : {crop} ({conf}% confidence)')
    print('Top 3 predictions:')
    for c, p in top3:
        bar = '█' * int(p * 30)
        print(f'  {c:<10} {bar:<30} {p*100:.1f}%')
    return crop, conf

print('=== Test 1: Hot, wet, high-N field (Terai) ===')
predict_crop(nitrogen=90, phosphorus=38, potassium=48,
             temperature=30, moisture=85, ph=6.2)

print()
print('=== Test 2: Cool, moderate moisture (Hill region) ===')
predict_crop(nitrogen=88, phosphorus=70, potassium=110,
             temperature=16, moisture=72, ph=6.0)

print()
print('=== Test 3: Warm, moderate conditions ===')
predict_crop(nitrogen=95, phosphorus=44, potassium=60,
             temperature=25, moisture=63, ph=6.3)

## 7. Save Model

In [ ]:
model_dir = '../website/models'
os.makedirs(model_dir, exist_ok=True)

joblib.dump(model, os.path.join(model_dir, 'crop_model.pkl'))
joblib.dump(le,    os.path.join(model_dir, 'label_encoder.pkl'))

model_size = os.path.getsize(os.path.join(model_dir, 'crop_model.pkl')) / 1024
print(f'Model saved     → website/models/crop_model.pkl  ({model_size:.1f} KB)')
print(f'Encoder saved   → website/models/label_encoder.pkl')
print()
print(f'Final Test Accuracy : {test_acc*100:.2f}%')
print(f'CV Accuracy         : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print(f'Features            : {FEATURES}')
print(f'Classes             : {list(le.classes_)}')